# Task 7: бот-помощник (hello / weather / currency / NoMatch)

В этой тетради реализован простой бот с 4 состояниями:
- `/hello` — приветствие пользователя;
- `/weather` — ответ по погоде;
- `/currency` — ответ по курсу валют;
- `/NoMatch` — фолбэк, если интент не распознан.

Добавлен тестовый набор фраз, чтобы проверить распознавание разных формулировок интентов.

In [1]:
%pip -q install requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random
import re
from dataclasses import dataclass

import requests

STATE_HELLO = "/hello"
STATE_WEATHER = "/weather"
STATE_CURRENCY = "/currency"
STATE_NOMATCH = "/NoMatch"

HELLO_PATTERNS = [
    r"\bпривет\b", r"\bздравствуй\b", r"\bдобрый\b", r"\bhello\b", r"\bhi\b"
]
WEATHER_PATTERNS = [
    r"погод", r"температур", r"дожд", r"снег", r"ветер", r"weather", r"жарк", r"холод"
]
CURRENCY_PATTERNS = [
    r"курс", r"валют", r"доллар", r"евро", r"рубл", r"usd", r"eur", r"rub", r"cny", r"gbp"
]

CITY_ALIASES = {
    "москва": "Moscow",
    "санкт-петербург": "Saint Petersburg",
    "петербург": "Saint Petersburg",
    "новосибирск": "Novosibirsk",
    "екатеринбург": "Yekaterinburg",
    "казань": "Kazan",
}

CURRENCY_ALIASES = {
    "доллар": "USD",
    "доллара": "USD",
    "долларов": "USD",
    "евро": "EUR",
    "рубль": "RUB",
    "рубля": "RUB",
    "рублей": "RUB",
    "юань": "CNY",
    "фунт": "GBP",
    "usd": "USD",
    "eur": "EUR",
    "rub": "RUB",
    "cny": "CNY",
    "gbp": "GBP",
}

@dataclass
class BotReply:
    state: str
    text: str


def _contains_any(text: str, patterns: list[str]) -> bool:
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in patterns)


def _extract_city(text: str) -> str:
    lowered = text.lower()
    for ru_name, normalized in CITY_ALIASES.items():
        if ru_name in lowered:
            return normalized
    match = re.search(r"\bв\s+([a-zA-Zа-яА-Я\-]+)", text)
    if match:
        return match.group(1)
    return "Moscow"


def _extract_currency_pair(text: str) -> tuple[str, str]:
    lowered = text.lower()
    codes = []

    for token in re.findall(r"[a-zA-Zа-яА-Я]+", lowered):
        code = CURRENCY_ALIASES.get(token)
        if code:
            codes.append(code)

    if len(codes) >= 2:
        return codes[0], codes[1]
    if len(codes) == 1:
        return codes[0], "RUB"
    return "USD", "RUB"


def handle_hello() -> BotReply:
    variants = [
        "Привет! Я бот-помощник. Могу подсказать погоду и курсы валют.",
        "Здравствуйте! Спросите меня о погоде или курсе валют."
    ]
    return BotReply(state=STATE_HELLO, text=random.choice(variants))


def handle_weather(user_text: str) -> BotReply:
    city = _extract_city(user_text)
    try:
        response = requests.get(f"https://wttr.in/{city}?format=j1", timeout=15)
        response.raise_for_status()
        payload = response.json()["current_condition"][0]
        temp_c = payload.get("temp_C", "?")
        feels_like = payload.get("FeelsLikeC", "?")
        desc = payload.get("weatherDesc", [{}])[0].get("value", "без описания")
        text = (
            f"Погода в {city}: {desc.lower()}, температура {temp_c}°C, "
            f"ощущается как {feels_like}°C."
        )
    except Exception:
        text = "Не получилось получить погоду сейчас. Попробуйте повторить запрос чуть позже."
    return BotReply(state=STATE_WEATHER, text=text)


def handle_currency(user_text: str) -> BotReply:
    base, quote = _extract_currency_pair(user_text)
    if base == quote:
        return BotReply(
            state=STATE_CURRENCY,
            text=f"{base}/{quote}: 1.0 (это одна и та же валюта).",
        )

    try:
        response = requests.get(f"https://open.er-api.com/v6/latest/{base}", timeout=15)
        response.raise_for_status()
        payload = response.json()
        rate = payload["rates"][quote]
        text = f"Курс {base}/{quote}: 1 {base} = {rate:.4f} {quote}."
    except Exception:
        text = "Не удалось получить курс валют прямо сейчас. Попробуйте позже."

    return BotReply(state=STATE_CURRENCY, text=text)


def handle_no_match() -> BotReply:
    variants = [
        "Я пока понимаю только запросы про погоду и курсы валют.",
        "Не понял запрос. Попробуйте спросить: 'погода в Москве' или 'курс доллара'."
    ]
    return BotReply(state=STATE_NOMATCH, text=random.choice(variants))


def route(user_text: str) -> BotReply:
    text = user_text.strip()
    lowered = text.lower()

    if _contains_any(lowered, HELLO_PATTERNS):
        return handle_hello()

    weather_hit = _contains_any(lowered, WEATHER_PATTERNS)
    currency_hit = _contains_any(lowered, CURRENCY_PATTERNS)

    if weather_hit and currency_hit:
        weather_pos = min((lowered.find(k) for k in ["погод", "температур", "weather"] if lowered.find(k) != -1), default=10**9)
        currency_pos = min((lowered.find(k) for k in ["курс", "валют", "доллар", "евро", "usd", "eur"] if lowered.find(k) != -1), default=10**9)
        return handle_weather(text) if weather_pos <= currency_pos else handle_currency(text)

    if weather_hit:
        return handle_weather(text)
    if currency_hit:
        return handle_currency(text)

    return handle_no_match()


test_phrases = [
    "Привет!",
    "Здравствуй, бот",
    "Какая погода в Москве?",
    "Что с температурой в Санкт-Петербурге",
    "Курс доллара",
    "Сколько сейчас EUR в RUB?",
    "расскажи анекдот",
]

for phrase in test_phrases:
    answer = route(phrase)
    print(f"User: {phrase}")
    print(f"State: {answer.state}")
    print(f"Bot: {answer.text}\n")

User: Привет!
State: /hello
Bot: Привет! Я бот-помощник. Могу подсказать погоду и курсы валют.

User: Здравствуй, бот
State: /hello
Bot: Привет! Я бот-помощник. Могу подсказать погоду и курсы валют.

User: Какая погода в Москве?
State: /weather
Bot: Погода в Москве: partly cloudy, температура 5°C, ощущается как 1°C.

User: Что с температурой в Санкт-Петербурге
State: /weather
Bot: Погода в Saint Petersburg: partly cloudy, температура 8°C, ощущается как 6°C.

User: Курс доллара
State: /currency
Bot: Курс USD/RUB: 1 USD = 76.0321 RUB.

User: Сколько сейчас EUR в RUB?
State: /currency
Bot: Курс EUR/RUB: 1 EUR = 89.8892 RUB.

User: расскажи анекдот
State: /NoMatch
Bot: Я пока понимаю только запросы про погоду и курсы валют.



## JAICP-версия

Готовый сценарий для платформы JAICP находится в файле:
- `task-7-jaicp/main.sc`

Тестовый клиент для Chat API:
- `task-7-jaicp/chat_api_test.py`

Как запустить:
1. В JAICP откройте проект и вставьте содержимое `main.sc`.
2. Опубликуйте Chat API канал и скопируйте токен.
3. В `chat_api_test.py` подставьте токен в `CHANNEL_TOKEN`.
4. Запустите: `python3 task-7-jaicp/chat_api_test.py`.